[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/thermocouple/thermocouple.ipynb)

# Thermocouple

This example computes the two effects of a thermocouple, a wire of two metals joined at one end: heat at the junction gives a voltage between the wire's free ends, the Seebeck effect, and a current through the junction cools or warms it, the Peltier effect. An electron holds a charge and an entropy, so a fall of the electrochemical potential pushes it by its charge and a fall of temperature pushes it by its entropy, and wherever it drifts it takes both along. The electrons of the two metals hold different entropies. With the junction warmer than the ends and no current passing, each metal builds up the voltage that holds its own electrons still against the fall of temperature, and the difference between the two metals' voltages stands between the ends. With a current passing, the electrons bring one entropy to the junction and take another away, and the junction cools or warms by the difference, according to the current's sense. The two effects are tied to each other: a unit of heat a second put in at the junction raises the voltage by as much as a unit of current lowers the junction's temperature.

The potentials are modelled as one vector, the electrochemical potential along time and the temperature along a direction of its own for entropy, and their fall is one bivector, `drive = -(gradient ^ potentials)`. The direction for entropy is degenerate, since nothing measures an entropy against a length, so no dot product reads it: every reading is the regressive product of a potential or a drive with what it reads, as the power is, `drive & flux`. A carrier's parcel, its charge and its entropy, is one element too, and the carrier's velocity joined with its parcel is the flux it makes. As a map from velocities to fluxes that is the advection, `Spatial.regressive(parcel)`, and its adjugate takes a drive to the force on the carrier, so the push on a carrier and the flux it makes are one map, read forwards and backwards. With a mobility between the two, a metal's response is `advection(mobility(advection.adjugate()))`, which is its own adjugate because the mobility is: at one place, that ties the two effects. Over the wire, the wire's conductance is the response between the gradient and its adjugate, `gradient.adjugate()(response(gradient))`, its own adjugate again, which ties the two effects of the whole wire.

In the notation of irreversible thermodynamics, the same model reads as two linear laws for the densities of the electric current and of the heat current, $\mathbf J = \sigma(\mathbf E - S\nabla T)$ and $\mathbf q = \Pi\mathbf J - \kappa\nabla T$, with the Seebeck coefficient $S$ and the Peltier coefficient $\Pi$, and Kelvin's relation $\Pi = TS$ between them, Onsager's reciprocal relation for this pair of coefficients.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_animation
from examples.util.topology import Topology
from examples.thermodynamics.thermocouple import render, scenarios

# Planar spacetime, its degenerate w weighing a place, and the entropy a parcel holds: a direction of its own,
# degenerate, since nothing measures an entropy against a length or a time.
ga = Algebra("x+y+t-w0") * Algebra("s0")
# The plane of space, and what a parcel holds, charge along time and entropy.
space = Algebra("x+y+")
amounts = Algebra("t-s0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# What a parcel holds, charge along time and entropy; and a direction through space, a carrier's velocity.
Parcel = Plane.derive.within(amounts).derive.dual                             # Antivector-
Spatial = Plane.derive.within(space).derive.dual                              # Antivector-
# A force on a carrier, which reads its velocity; the potentials, the electrochemical potential along t and the
# temperature along s; and their drive.
Force = Plane.derive.within(space)                                            # Vector+
Potential = Plane.derive.within(amounts)                                      # Vector+
Drive = ga.gatype.bivector().derive.touching(space, amounts)                  # Bivector+
# A unit of charge, a direction along time, and a unit of entropy.
time = -mv.t.dual()                                                           # [] Parcel-
entropy = mv.s.dual()                                                         # [] Parcel-
np.set_printoptions(precision=3, suppress=True)

## A metal's response

A metal has two kinds of carrier here: its electrons, which hold a charge and an entropy, and the vibrations of its lattice, which hold entropy and no charge. A carrier's parcel is its share of both, its charge times the unit of charge plus its entropy times the unit of entropy. A carrier's velocity joined with its parcel is the flux it makes; multiplying that product by `Spatial`, the space of all velocities, gives the advection as a map over velocities. The adjugate of a map runs the other way and keeps the pairing, `drive & advection(velocity) == advection.adjugate()(drive) & velocity`: the advection's adjugate takes a drive to the force on the carrier, which reads the carrier's velocity as the power the drive spends on it. The mobility takes that force to a velocity, the same along every direction of space, `Force.dual()`, times the velocity a unit force gives, with the kind's density counted in. The three maps composed, and summed over the kinds, are the metal's response.

In [ ]:
# The two kinds of carrier, electrons and the lattice's vibrations: the charge each holds; the entropy each holds, in
# the first metal and in the second; and its mobility, the velocity a unit force gives it, times its density.
charges = np.array([-1, 0])                                                   # [kinds]
entropies = np.array([[2.0, 1.0], [0.5, 1.0]])                                # [metals, kinds]
mobilities = np.array([1.0, 1.0])                                             # [kinds]

In [ ]:
# Each kind's parcel in each metal, its charge along time and its entropy.
parcels = charges * time + entropies * entropy                               # [metals, kinds] Parcel-
# A carrier's velocity joined with its parcel is the flux it makes; the adjugate of that map takes a drive to the
# force on the carrier.
advection = Spatial.regressive(parcels)                                       # [metals, kinds] Antibivector- <- Antivector-
mobility = mv.scalar(mobilities[:, None]) * Force.dual()                      # [kinds] Antivector- <- Vector+
responses = advection(mobility(advection.adjugate())).sum(axis=-1)            # [metals] Antibivector- <- Bivector+
# The electrochemical potential and the temperature each rising along x at unit slope, and their drives, their
# falls; the flux each drives in the first metal, read for charge and for entropy by the same two drives.
drives = -(mv.x ^ stack((mv.t, mv.s)))                                        # [drives] Drive+
moved = drives[:, None] & responses[0](drives)                               # [drives, drives] Scalar+

print("charge and entropy moved by a fall of the electrochemical potential:", moved.to_array()[:, 0])
print("charge and entropy moved by a fall of the temperature:              ", moved.to_array()[:, 1])

A unit fall of the electrochemical potential moves one unit of charge, and with it minus two of entropy: the electrons run up the potential, against the charge they hold, and take their entropy along. A unit fall of the temperature moves the same minus two units of charge, and five of entropy, four with the electrons and one through the lattice. The two cross readings are one number, because the response is its own adjugate.

## The hairpin

The wire is bent into a hairpin, a leg of each metal joined across the top, and comes from the scenario as a mesh of triangles, its vertices points of weight one. Its two feet are clamped to one cold block, and the heat is put in along the top of the junction. On each triangle, a corner's simplex gradient is the slope of the function that is one at that corner and nothing along the side across: that side, joined with time and entropy, along which the potentials do not vary off the mesh, over that join closed by the corner.

In [ ]:
# The hairpin: its triangles, its vertices, each triangle's metal, its feet, the first metal's and the second's, and
# the vertices heated along the top of the junction.
topology, vertices, metals, high, low, heated = scenarios.hairpin(scenarios.spacing)
corners = topology.incidence[2, 0] * vertices                                 # [corners] Point-[faces]
facing = topology.off[2, 0] * vertices                                        # [2, corners] Point-[faces]
sides = facing[0] & facing[1]                                                 # [corners] Antibivector-[faces]
# Each corner's simplex gradient, its slope across space kept and its value at the origin, read along w, dropped; and
# each triangle's area, half its corners' join along time and entropy.
gradients = ((sides & time & entropy) / (sides & time & entropy & corners)).cast(Force)   # [corners] Vector+[faces]
areas = (corners[0] & corners[1] & corners[2] & time & entropy).norm() / 2    # Scalar+[faces]
render.draw_hairpin(topology, vertices, metals, (high, low), scenarios.resolution);

The first metal is the left leg and the second the right; the line across the top is the junction between them, and the gold feet are where the hairpin meets the block.

## The hairpin's conductance

The potentials at a triangle's corners, each on its corner's simplex gradient, give the potentials' gradient wedged with them on the triangle; over all triangles that is one map, the gradient, from the potentials at the vertices to a bivector on each triangle. Its adjugate takes a flux on each triangle to what the flux carries out of each vertex's cell, charge and entropy. With each triangle's response between the two, times its area, the composition is the hairpin's conductance: what each cell loses as the potentials rise at each vertex. A cell also takes up entropy as its temperature rises, in proportion to its area, and no charge, so the temperature takes time to spread while the electrochemical potential follows it at once. The second foot's cells lose entropy to the cold block in the same form, in proportion to their temperature, which holds them near the block's temperature and leaves their voltage free; the first foot is held at its potentials in equilibrium.

In [ ]:
# The entropy a unit of area takes up for each unit its temperature rises, and the entropy the block takes from a unit
# of area of the second foot a unit of time, for each unit of its temperature.
capacity, clamping = scenarios.capacity, scenarios.clamping

In [ ]:
# The gradient: the potentials at the vertices to their gradient wedged with them on each triangle.
at_corners = topology.incidence[2, 0]                                         # [corners] Scalar+[faces <- vertices]
gradient = (spdiag(gradients ^ Potential) * at_corners).sum(axis=0)           # (Bivector+ <- Vector+)[faces <- vertices]
# What each cell loses as the potentials rise: each triangle's response, as its metal's, between the gradient and its
# adjugate.
conducting = gradient.adjugate()((areas * responses[metals].field())(gradient))   # (Antivector- <- Vector+)[vertices <- vertices]
# A temperature read off the potentials, as that much entropy; each vertex's cell, a third of each triangle around
# it; what the cells take up, and what the second foot's cells lose to the block.
warmed = entropy * (Potential & entropy)                                      # [] Antivector- <- Vector+
cells = (~at_corners * (areas / 3)).sum(axis=0)                               # Scalar+[vertices]
on_block = mv.scalar(np.isin(np.arange(topology.n_vertices), low)[:, None]).field()   # Scalar+[vertices]
storing = spdiag(cells * capacity * warmed)                                   # (Antivector- <- Vector+)[vertices <- vertices]
losing = conducting + spdiag(cells * on_block * clamping * warmed)            # (Antivector- <- Vector+)[vertices <- vertices]

## Heated, and driven by a current

From equilibrium, the hairpin is driven two ways, apart. Heated, a unit of entropy a unit of time is put in along the top of the junction. Driven by a current, a unit of charge a unit of time leaves through the second foot, and enters through the first, which is held. At each step every free cell takes up what is put into it less what it loses, solved at the step's end. The model is linear response: the entropy the drives produce, which grows as their square, is left out. This cell and the next take about three seconds, most of it drawing.

In [ ]:
# The two drives, the entropy put in at the junction a unit of time and the current; the frames, their duration in
# milliseconds, and the time between them.
heating, current = scenarios.heating, scenarios.current
n_frames, frame_duration_ms, time_step = scenarios.n_frames, scenarios.frame_duration_ms, scenarios.time_step

In [ ]:
def settling(sources: Parcel) -> Iterator[Potential]:
    """The hairpin's potentials `Potential+[vertices]` at each frame, from equilibrium, with this put into each cell
    a unit of time `Parcel-[vertices]` and the first foot held."""
    free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), high), 0)   # Scalar+[free <- vertices]
    stepping = free * (losing + storing / time_step) * ~free                  # (Antivector- <- Vector+)[free <- free]
    # What is put in, and the potentials, in their full types: charge and entropy, and both potentials.
    nothing = 0 * ((mv.t + mv.s) & sources)                                   # Scalar+[vertices]
    sources = sources + nothing * (time + entropy)                            # Parcel-[vertices]
    potentials = nothing * (mv.t + mv.s)                                      # Potential+[vertices]
    for _ in range(n_frames):
        yield potentials
        potentials = ~free * stepping.solve(free * (sources + storing(potentials) / time_step))


# The heating shared evenly between the heated vertices, and the current leaving shared between the second foot's.
shares = np.zeros((2, topology.n_vertices))
shares[0, heated], shares[1, low] = 1 / len(heated), 1 / len(low)
at_heated, at_low = mv.scalar(shares[..., None]).field()                      # Scalar+[vertices], Scalar+[vertices]
frames = list(zip(settling(at_heated * heating * entropy), settling(at_low * -current * time)))
show_animation(render.animate(topology, vertices, metals, (high, low), scenarios.resolution, frames[-1], frames),
               "thermocouple", frame_duration_ms, n_frames=n_frames)

On the left the hairpin is heated at its junction, and on the right it is driven by the current. Each is tinted by its temperature, grey in equilibrium, red where warmer and blue where cooler, with the level lines of its electrochemical potential over it. Heated, the warmth spreads from the junction down both legs. No current passes, so in each leg the potential falls with the temperature by the entropy that leg's electrons hold for each unit of their charge: the level lines crowd in the first metal and thin out in the second, and the difference is the voltage between the feet. Driven by the current, the electrons arrive at the junction through the second metal with little entropy and leave through the first with more, and the junction cools; the lattice conducts warmth up both legs from the block.

## The two effects against time

A voltmeter reads the electrochemical potential at the second foot above the first's, which is held at nothing, and a thermometer reads the temperature along the top of the junction, each a mean over its vertices, off the states that the animation drew. Heated, the voltage is taken per unit of heating; driven by the current, the fall of the junction's temperature per unit of current.

In [ ]:
warmed_states = stack([potentials for potentials, _ in frames])              # [frames] Potential+[vertices]
driven_states = stack([potentials for _, potentials in frames])              # [frames] Potential+[vertices]
voltage = (topology.reading(low, 0) * (warmed_states & time)).batch().mean(axis=-1) / heating       # [frames] Scalar+
cooling = -(topology.reading(heated, 0) * (driven_states & entropy)).batch().mean(axis=-1) / current   # [frames] Scalar+
render.draw_readings(time_step * np.arange(n_frames), voltage, cooling);

The line is the heated hairpin's voltage per unit of heating, and the dots are the driven hairpin's cooling per unit of current. They coincide at every moment, not only once settled: the conductance and the storage are each their own adjugate, so heat put in at the junction reads at the foot as a current drawn at the foot reads at the junction.

In [ ]:
# checks
# The push and the flux are one map read both ways: a drive's reading of the flux a velocity makes is the force's
# reading of the velocity. Each metal's response is its own adjugate, and its dissipation is positive.
velocity = mv.x.dual()                                                        # [] Spatial-
force = advection.adjugate()(drives[1])                                         # [metals, kinds] Force+
np.testing.assert_allclose((drives[1] & advection(velocity)).to_array(), (force & velocity).to_array(), atol=1e-12)
np.testing.assert_allclose(responses.adjugate().kernel, responses.kernel, atol=1e-12)
assert (np.diagonal(moved.to_array()) > 0).all()
# A carrier is pushed down its potentials: under each fall, a unit of charge and a unit of entropy drift against x.
units = stack((time, entropy))                                                # [drives] Parcel-
assert ((mv.x & Force.dual()((Spatial & units).adjugate()(drives))).to_array() < 0).all()
# The hairpin's conductance is its own adjugate too: the heated state's reading of what the driven state loses is the
# driven state's reading of what the heated state loses.
warmed_last, driven_last = frames[-1]
np.testing.assert_allclose((warmed_last & losing(driven_last)).sites.sum().to_array(),
                           (driven_last & losing(warmed_last)).sites.sum().to_array(), rtol=1e-9)
# Heat's voltage and the current's cooling coincide at every frame, and both have all but settled by the last.
np.testing.assert_allclose(voltage.to_array(), cooling.to_array(), atol=1e-9)
assert voltage.to_array()[-1] > 0
assert abs(voltage.to_array()[-1] - voltage.to_array()[-5]) < 2e-2 * voltage.to_array()[-1]